In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!pip install transformers
!pip install sentencepiece
!pip install datasets
#!pip install langdetect # Only for cleaning dataset

## Local Inference on GPU 
Model page: https://huggingface.co/vesteinn/ScandiBERT

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/vesteinn/ScandiBERT)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("kaggle-scandibert")

import torch
import math
import re
from huggingface_hub import hf_hub_download
from transformers import XLMRobertaTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset, concatenate_datasets

model_name = "vesteinn/ScandiBERT" # Considering no-faroese variant for slight tokenization difference
labels = ["Norwegian", "Non-Scandinavian"]
id2label = {i: l for i, l in enumerate(labels)}
label2id = {l: i for i, l in enumerate(labels)}

# Hard-coded because tokenizer.json is broken
vocab_path = hf_hub_download(repo_id=model_name, filename="sentencepiece.bpe.model")
tokenizer = XLMRobertaTokenizer(vocab_file=vocab_path)

SENTENCES_PER_CLASS = 3000

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels = len(labels),
    id2label = id2label,  
    label2id = label2id,
    ignore_mismatched_sizes = True,
    problem_type = "multi_label_classification"
)

def get_opus_sentences(config, lang_code, num_sentences=SENTENCES_PER_CLASS):
    print(f"Downloading {lang_code} from {config}...")
    ds = load_dataset("Helsinki-NLP/opus-100", config, split="train")
    ds = ds.filter(lambda sentence: is_dataset_clean(sentence["translation"][lang_code]))
    ds = ds.shuffle(seed=42).select(range(num_sentences))
    return [item[lang_code] for item in ds["translation"]]

def is_dataset_clean(text, instance_threshold=2, min_length=4, max_length=20):
    bad_char_regex = r"[@#$%^&*_=+\[\]{};':\"<>/`~\\|]"
    bad_format_regex = r'[A-Z][a-z]+[A-Z]|%\s*\d|https?://'
    if not isinstance(text, str):
        return False
    if len(text.strip()) < max_length or len(text.split()) < min_length:
        return False
    if len(re.findall(bad_char_regex, text)) >= instance_threshold:
        return False
    if re.search(bad_format_regex, text):
        return False
    return True        

def sent_based_int(percent, sent_const=SENTENCES_PER_CLASS): # Get an integer that's a % of SENTENCES_PER_CLASS
    return int(math.ceil(sent_const * percent))

sample_norwegian_all = get_opus_sentences("en-nb", "nb", sent_based_int(0.75)) + get_opus_sentences("en-nn", "nn", sent_based_int(0.25))
nonscandi_configs = [("en-es", "en"), ("en-es", "es"), ("de-en", "de"), ("en-nl", "nl"), ("af-en", "af")]
sample_nonscandi_all = []
for config, code in nonscandi_configs:
    sample_nonscandi_all.extend(get_opus_sentences(config, code, sent_based_int(1 / len(nonscandi_configs))))
print(f"Norwegian corpora collected: {len(sample_norwegian_all)} sentences.")
print(f"Non-Scandinavian corpora collected: {len(sample_nonscandi_all)} sentences.")

text = "Dette er en norsk setning."
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
print("\nZero-Shot Classification Probabilities: ")
probs = torch.sigmoid(outputs.logits)
for label, prob in zip(labels, probs[0]): # True "zero-shot", random classification test
    print(f"{label}: {prob.item():.4f}")

sentencepiece.bpe.model:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/523 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: vesteinn/ScandiBERT
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


README.md: 0.00B [00:00, ?B/s]

... (40 outputs left out)

Generating train split:   0%|          | 0/275512 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/275512 [00:00<?, ? examples/s]

Norwegian corpora collected: 3000 sentences.
Non-Scandinavian corpora collected: 3000 sentences.

Zero-Shot Classification Probabilities: 
Norwegian: 0.4344
Non-Scandinavian: 0.4580


In [3]:
from datasets import Dataset, concatenate_datasets, Sequence, Value, Features
from sklearn.model_selection import train_test_split

eval_ds_split = 0.15

norwegian_train, norwegian_eval = train_test_split(
    sample_norwegian_all, test_size=eval_ds_split, random_state=42
)
nonscandi_train, nonscandi_eval = train_test_split(
    sample_nonscandi_all, test_size=eval_ds_split, random_state=42
)

label_features = Features({"text": Value("string"), "labels": Sequence(Value("float32"))})

dataset_norwegian_train = Dataset.from_dict({"text": norwegian_train, "labels": [[1.0, 0.0]] * len(norwegian_train)}, features=label_features)
dataset_nonscandi_train = Dataset.from_dict({"text": nonscandi_train, "labels": [[0.0, 1.0]] * len(nonscandi_train)}, features=label_features)
train_dataset = concatenate_datasets([dataset_norwegian_train, dataset_nonscandi_train]).shuffle(seed=42)

dataset_norwegian_eval = Dataset.from_dict({"text": norwegian_eval, "labels": [[1.0, 0.0]] * len(norwegian_eval)}, features=label_features)
dataset_nonscandi_eval = Dataset.from_dict({"text": nonscandi_eval, "labels": [[0.0, 1.0]] * len(nonscandi_eval)}, features=label_features)
eval_dataset = concatenate_datasets([dataset_norwegian_eval, dataset_nonscandi_eval]).shuffle(seed=42)

print(train_dataset[:20])

def tokenize_dataset(select_dataset):
    return tokenizer(select_dataset["text"], truncation=True, max_length=512)

print(f"{len(train_dataset)} Training samples | {len(eval_dataset)} Eval samples")

train_dataset = train_dataset.map(tokenize_dataset, batched=True)
eval_dataset = eval_dataset.map(tokenize_dataset, batched=True)
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

{'text': ["Ik moet nog extra werken na m'n reis naar Afrika.", 'Som en følge av Nouvelles Frontièressaken (1986), der Domstolen la til grunn at konkurransereglene i Felles skapets traktater kommer til anvendelse på flypriser, har Domstolens avgjørelser på dette område for eksempel gjort det mulig å oppnå gunstigere prisbetingelser for flypassasjerer. sasjerer.', 'Ik heb Maggie bij Robin achtergelaten.', 'Zou het makkelijk zijn hen te vinden? Natuurlijk wel.', 'Schäden am Ohr wie perforierte Trommelfell können Sie besser über internen Sounds machen', 'The window I authorized town funds to repair after the storm blew it out?', 'After her release on 17 August 2000 she was first admitted to a clinic because of her injuries, where she received notice of the visa having been issued.', 'Disse internettbaserte tjenestene retter seg mot unge mennesker og fokuserer på problemer relatert til alkohol og cannabis.', 'You want to kill us before, but now you said that you liked Shun.', 'En overvåker 

Map:   0%|          | 0/5100 [00:00<?, ? examples/s]

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

In [4]:
import re
from transformers import TrainingArguments, Trainer
from torch.nn.utils.rnn import pad_sequence

def brute_force_collator(features): # Brute force AGAIN
    input_ids = pad_sequence(
        [f["input_ids"] for f in features],
        batch_first=True,
        padding_value=tokenizer.pad_token_id
    )
    attention_mask = pad_sequence(
        [f["attention_mask"] for f in features],
        batch_first=True,
        padding_value=0
    )
    labels = torch.stack([f["labels"] for f in features])
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

training_args = TrainingArguments(
    output_dir = "./scandiprob_checkpoints",
    learning_rate = 2e-5,                    # = 0.00002 learning rate
    per_device_train_batch_size = 8,         # Adjust to 4 if Kaggle throws an OutOfMemory (OOM) error
    gradient_accumulation_steps = 1,
    num_train_epochs = 6,                   
    weight_decay = 0.01,
    fp16 = False,
    warmup_steps = sent_based_int(0.01),
    report_to = "none",
    dataloader_pin_memory = False,
    logging_steps = sent_based_int(0.1),
    eval_strategy = "epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=brute_force_collator
)

def nonscandi_penalty(text): # Regex for negative class
    if not text:
        return 0.0

    scandi_keyboard = r"[a-zA-ZæøåÆØÅäöÄÖéÉ0-9 !@#$%^&*()\-_=+\[\]{};':\",.<>?/`~\\|]"
    scandi_matches = re.findall(scandi_keyboard, text)
    nonscandi_percent = (1 - (len(scandi_matches) / len(text)))
    return nonscandi_percent

print("Training...")
trainer.train()
trainer.save_model("./scandiprob_current")
print("Training complete and model saved!")


test_text = "Dette er en norsk setning."
test_inputs = tokenizer(test_text, return_tensors="pt")
test_inputs = {k: v.to(model.device) for k, v in test_inputs.items()}
model.eval()

with torch.no_grad():
    outputs = model(**test_inputs)

print("\n")
raw_probs = torch.sigmoid(outputs.logits)[0]

nonscandi_ratio = nonscandi_penalty(test_text)


final_probs = []

for i, label in enumerate(labels):
    prob = raw_probs[i].item()
    
    if label == "Norwegian":
        adjusted = prob * (1.0 - nonscandi_ratio) 
    elif label == "Non-Scandinavian": # Scale up/down by % of non-Scandi chars
        adjusted = prob + ((1.0 - prob) * nonscandi_ratio)
        
    final_probs.append(adjusted)

skew_effect = (raw_probs != final_probs)
if skew_effect:
    print("\nPre-Adjustment Probabilities: ")
    for label, prob in zip(labels, raw_probs): # Print pre-regex-skew results!
        print(f"{label}: {prob:.4f}")
print("\nPost-Adjustment Probabilities: " if skew_effect else "\nPost-Training Probabilities")
for label, prob in zip(labels, final_probs): # Print post-regex-skew results!
    print(f"{label}: {prob:.4f}")

Training...


Epoch,Training Loss,Validation Loss
1,0.695961,0.693316
2,0.694630,0.694154
3,0.694919,0.693435
4,0.694072,0.693119
5,0.693533,0.692240
6,0.695023,0.690296


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (2 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete and model saved!



Pre-Adjustment Probabilities: 
Norwegian: 0.5874
Non-Scandinavian: 0.4386

Post-Adjustment Probabilities: 
Norwegian: 0.5874
Non-Scandinavian: 0.4386
